# Chapter 3: Kinetics — Mass, Coriolis, and Hydrodynamic Damping

In Chapter 2, we focused entirely on **Kinematics** (the geometry of motion). Now, we step into **Kinetics**, where we analyze the forces and torques that cause a vessel to move. 

Fossen structures the full 6-DOF nonlinear equations of motion for a marine vehicle using a highly elegant matrix framework:

$$M \dot{\nu} + C(\nu)\nu + D(\nu)\nu + g(\eta) = \tau$$

Where:
* **$M$** : The total **Mass & Inertia Matrix** (Resistance to acceleration).
* **$C(\nu)$** : The **Coriolis & Centripetal Matrix** (Inertial forces felt within a rotating frame).
* **$D(\nu)$** : The **Damping Matrix** (Water resistance and skin friction).
* **$g(\eta)$** : The **Restoring Forces Vector** (Gravity and buoyancy).
* **$\tau$** : The **Control Vector** (Thrust and torque from thrusters/rudders).

---

## 1. The Rigid-Body Mass Matrix ($M_{RB}$)

Before dealing with water, we must map the dry mass of the vessel. For a 3D rigid body, mass is a $6 \times 6$ matrix. If we place our coordinate origin exactly at the vehicle's **Center of Gravity (CG)**, the cross-coupling terms drop out, leaving a clean, diagonal-blocked structure:

$$M_{RB} = \begin{bmatrix} m I_{3 \times 3} & 0_{3 \times 3} \\ 0_{3 \times 3} & I_g \end{bmatrix}$$

Let's look at how a vessel's physical size and mass change its rotational acceleration profile under a fixed thruster torque.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def simulate_rigid_inertia(mass_tonnes, length_meters, thruster_force_kilonewtons):
    """
    Computes the vessel's yaw moment of inertia (Iz) using a standard rectangle hull
    approximation and calculates the resulting angular acceleration from a maneuvering thruster.
    """
    m = mass_tonnes * 1000.0          # Tonnes to kg
    L = length_meters                 # Meters
    tau_yaw = thruster_force_kilonewtons * 1000.0 * (L / 2.0)  # Torque = Force * distance to CG
    
    # Approximate Yaw Moment of Inertia (Iz) -> assuming beam/width is roughly 20% of length
    width = L * 0.2
    Iz = (1.0 / 12.0) * m * (L**2 + width**2)
    
    # Newton's Second Law for Rotation: Alpha = Torque / Inertia
    alpha_yaw = tau_yaw / Iz
    alpha_deg = np.degrees(alpha_yaw)
    
    # Time domain tracking
    t = np.linspace(0, 5, 200)
    yaw_rate = alpha_deg * t  # Angular Velocity (r) over time
    
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(t, yaw_rate, color='darkorange', linewidth=3, label='Yaw Rate (r)')
    
    ax.set_title("Rigid-Body Kinetics: Rotational Acceleration Profile", fontsize=12)
    ax.set_xlabel("Time Elapsed (Seconds)", fontsize=10)
    ax.set_ylabel("Yaw Angular Velocity (Degrees / Second)", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    text_info = (
        f"Vessel Mass (m)      = {m:,.0f} kg\n"
        f"Computed Inertia (Iz) = {Iz:,.0f} kg·m²\n"
        f"Applied Yaw Torque    = {tau_yaw:,.0f} N·m\n\n"
        f"Angular Acceleration  = {alpha_deg:.4f}°/s²"
    )
    ax.text(0.2, ax.get_ylim()[1]*0.65, text_info, fontsize=11, family='monospace',
            bbox=dict(boxstyle='round', facecolor='whitesmoke', edgecolor='gray', alpha=0.9))
    
    plt.show()

print("MANIPULATE VESSEL INERTIA TO OBSERVE REACTION DYNAMICS:")
interact(simulate_rigid_inertia,
         mass_tonnes=widgets.FloatSlider(min=10.0, max=500.0, step=10.0, value=100.0, description='Mass (T):'),
         length_meters=widgets.FloatSlider(min=5.0, max=100.0, step=5.0, value=40.0, description='Length (m):'),
         thruster_force_kilonewtons=widgets.FloatSlider(min=1.0, max=50.0, step=1.0, value=10.0, description='Thruster (kN):'));

MANIPULATE VESSEL INERTIA TO OBSERVE REACTION DYNAMICS:


interactive(children=(FloatSlider(value=100.0, description='Mass (T):', max=500.0, min=10.0, step=10.0), Float…

---

## 1.5 The Newton-Euler Formulation

To build the rigid-body mass ($M_{RB}$) and Coriolis ($C_{RB}$) matrices, Fossen applies Newton's Second Law for linear momentum and Euler's equations for angular momentum. 

When tracking these vectors about an arbitrary body-fixed origin $O$ that is **not** aligned with the Center of Gravity ($CG$), we define the spatial offset to the CG as the lever arm vector:

$$r_g = [x_g, y_g, z_g]^T$$

### The Core Vector Equations
The total forces ($f_b$) and moments ($m_b$) acting on the rigid body in the moving BODY frame are formulated as:

$$f_b = m \left( \dot{v}_b + \omega_b \times v_b + \dot{\omega}_b \times r_g + \omega_b \times (\omega_b \times r_g) \right)$$

$$m_b = I_g \dot{\omega}_b + \omega_b \times (I_g \omega_b) + m \cdot r_g \times \left( \dot{v}_b + \omega_b \times v_b \right)$$

Where:
* $v_b = [u, v, w]^T$ is the linear velocity vector in the BODY frame.
* $\omega_b = [p, q, r]^T$ is the angular velocity vector in the BODY frame.
* $I_g$ is the inertia tensor calculated directly about the Center of Gravity.

### Matrix Mapping: Generating $M_{RB}$
By expanding those cross products ($\times$) using skew-symmetric cross-product matrices ($S(\cdot)$), the Newton-Euler equations map perfectly into Fossen's $6 \times 6$ rigid-body mass matrix format:

$$M_{RB} = \begin{bmatrix}
m I_{3 \times 3} & -m S(r_g) \\
m S(r_g) & I_g - m S^2(r_g)
\end{bmatrix}$$

Where the skew-symmetric matrix $S(r_g)$ acts as a matrix-based cross-product operator:

$$S(r_g) = \begin{bmatrix}
0 & -z_g & y_g \\
z_g & 0 & -x_g \\
-y_g & x_g & 0
\end{bmatrix}$$

Notice that if you manage to align your origin perfectly with the CG ($r_g = [0, 0, 0]^T$), the off-diagonal coupling terms $-m S(r_g)$ instantly vanish, leaving you with a decoupled, block-diagonal mass matrix!

In [1]:
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact

def interactive_newton_euler(mass_tonnes, length, beam, x_g, y_g, z_g):
    """
    Dynamically generates and displays the 6x6 Rigid-Body Mass Matrix (M_RB)
    to show how changing the Center of Gravity offset alters cross-coupling.
    """
    m = mass_tonnes * 1000.0  # Tonnes to kg
    height = beam * 0.6       # Assume height scales proportionally for inertia calculation
    
    # 1. Define the spatial lever arm vector from origin to CG: r_g
    r_g = np.array([x_g, y_g, z_g])
    
    # 2. Build the Skew-Symmetric Cross-Product Matrix S(r_g)
    S_rg = np.array([
        [  0.0, -z_g,  y_g],
        [  z_g,   0.0, -x_g],
        [ -y_g,  x_g,   0.0]
    ])
    
    # 3. Compute the Inertia Tensor (I_g) about the CG using a solid box model
    I_x = (1.0 / 12.0) * m * (beam**2 + height**2)
    I_y = (1.0 / 12.0) * m * (length**2 + height**2)
    I_z = (1.0 / 12.0) * m * (length**2 + beam**2)
    I_g = np.diag([I_x, I_y, I_z])
    
    # 4. Assemble the 6x6 matrix using block components derived from Newton-Euler
    M_RB = np.zeros((6, 6))
    
    # Top-Left: m * I_3x3 (Translational Mass)
    M_RB[0:3, 0:3] = m * np.eye(3)
    
    # Top-Right: -m * S(r_g) (Linear to Angular Coupling)
    M_RB[0:3, 3:6] = -m * S_rg
    
    # Bottom-Left: m * S(r_g) (Angular to Linear Coupling)
    M_RB[3:6, 0:3] = m * S_rg
    
    # Bottom-Right: I_g - m * S(r_g)^2 (Rotational Inertia about the chosen origin)
    M_RB[3:6, 3:6] = I_g - m * np.linalg.matrix_power(S_rg, 2)
    
    # --- Clean Printout ---
    np.set_printoptions(precision=1, suppress=True)
    print("=" * 75)
    print(f"            DYNAMIC NEWTON-EULER RIGID-BODY MASS MATRIX (M_RB)")
    print("=" * 75)
    print(f"  Vessel Mass : {m:,.0f} kg   |  Dimensions: L={length}m, B={beam}m")
    print(f"  CG Offset   : x_g={x_g:+.1f}m, y_g={y_g:+.1f}m, z_g={z_g:+.1f}m")
    print("-" * 75)
    
    # Labeling rows/columns for clarity
    labels = ['Surge', 'Sway ', 'Heave', 'Roll ', 'Pitch', 'Yaw  ']
    print("        " + "   ".join([f"{l}" for l in labels]))
    
    for i, row in enumerate(M_RB):
        # Format numbers nicely with spacing
        row_str = " ".join([f"{val:10.1f}" if val != 0 else f"{0.0:10.1f}" for val in row])
        print(f"{labels[i]} [{row_str} ]")
        
    print("-" * 75)
    
    # Structural Analysis Commentary
    if x_g == 0 and y_g == 0 and z_g == 0:
        print("✅ ORIGIN AT CG: The off-diagonal quadrants are empty. Decoupled 6-DOF tracking.")
    else:
        print("⚠️ OFFSET INDUCED CROSS-COUPLING:")
        couplings = []
        if x_g != 0: couplings.append("Sway-Yaw / Heave-Pitch")
        if y_g != 0: couplings.append("Surge-Yaw / Heave-Roll")
        if z_g != 0: couplings.append("Surge-Pitch / Sway-Roll")
        print(f"  Active physical cross-coupling channels: {', '.join(couplings)}")

print("ADJUST THE CENTER OF GRAVITY (CG) OFFSET TO TRANSFORM THE MASS MATRIX:")
interact(interactive_newton_euler,
         mass_tonnes=widgets.FloatSlider(min=10.0, max=500.0, step=10.0, value=100.0, description='Mass (T):'),
         length=widgets.FloatSlider(min=10.0, max=100.0, step=5.0, value=40.0, description='Length (m):'),
         beam=widgets.FloatSlider(min=2.0, max=20.0, step=1.0, value=8.0, description='Beam (m):'),
         x_g=widgets.FloatSlider(min=-10.0, max=10.0, step=0.5, value=0.0, description='x_g (Aft/Fwd):'),
         y_g=widgets.FloatSlider(min=-3.0, max=3.0, step=0.1, value=0.0, description='y_g (Port/Stbd):'),
         z_g=widgets.FloatSlider(min=-3.0, max=3.0, step=0.1, value=0.0, description='z_g (Keel/Deck):'));

ADJUST THE CENTER OF GRAVITY (CG) OFFSET TO TRANSFORM THE MASS MATRIX:


interactive(children=(FloatSlider(value=100.0, description='Mass (T):', max=500.0, min=10.0, step=10.0), Float…

---

## 2. Hydrodynamic Added Mass ($M_A$)

Because water is heavy (~800x denser than air), an accelerating hull forces the surrounding fluid to accelerate with it. To your guidance software, it feels like the vessel has suddenly become much heavier than its dry steel weight. We split our total mass matrix to account for this:

$$M = M_{RB} + M_A$$

Added mass is highly dependent on hull geometry. A slender submarine hull slices forward easily, meaning its forward added mass ($X_{\dot{u}}$) is tiny. However, if that same submarine tries to slide sideways (**Sway**), it pushes a massive wall of fluid, meaning its sideways added mass ($Y_{\dot{v}}$) can equal or exceed the weight of the actual ship.

In [2]:
def simulate_added_mass(added_mass_percentage):
    """
    Simulates how hydrodynamic added mass affects the effective weight of a 
    100-tonne vessel in both forward (Surge) and sideways (Sway) maneuvers.
    """
    dry_mass = 100000.0  # 100 tonnes in kg
    force = 20000.0      # 20 kN thruster force
    
    # Forward acceleration (low added mass, typically ~10% of dry mass)
    X_dot_u = dry_mass * 0.10
    total_mass_surge = dry_mass + X_dot_u
    accel_surge = force / total_mass_surge
    
    # Sideways acceleration (highly variable based on hull profile)
    Y_dot_v = dry_mass * (added_mass_percentage / 100.0)
    total_mass_sway = dry_mass + Y_dot_v
    accel_sway = force / total_mass_sway
    
    t = np.linspace(0, 4, 100)
    dist_surge = 0.5 * accel_surge * t**2
    dist_sway = 0.5 * accel_sway * t**2
    
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(t, dist_surge, color='seagreen', linewidth=3, label='Forward Surge (Fixed 10% Added Mass)')
    ax.plot(t, dist_sway, color='crimson', linewidth=3, linestyle='--', label=f'Sideways Sway ({added_mass_percentage}% Added Mass)')
    
    ax.set_title("Hydrodynamic Penalty: Surge vs. Sway Displacement over 4 Seconds", fontsize=12)
    ax.set_xlabel("Time (Seconds)", fontsize=10)
    ax.set_ylabel("Distance Traveled (Meters)", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left')
    
    text_info = (
        f"Dry Steel Mass      = {dry_mass/1000:.0f} Tonnes\n\n"
        f"SURGE CHANNEL (Forward):\n"
        f"  Added Mass (X_u)  = {X_dot_u/1000:.1f} Tonnes\n"
        f"  Effective Mass    = {total_mass_surge/1000:.1f} Tonnes\n"
        f"  Acceleration      = {accel_surge:.3f} m/s²\n\n"
        f"SWAY CHANNEL (Sideways):\n"
        f"  Added Mass (Y_v)  = {Y_dot_v/1000:.1f} Tonnes\n"
        f"  Effective Mass    = {total_mass_sway/1000:.1f} Tonnes\n"
        f"  Acceleration      = {accel_sway:.3f} m/s²"
    )
    ax.text(2.1, ax.get_ylim()[1]*0.08, text_info, fontsize=10, family='monospace',
            bbox=dict(boxstyle='round', facecolor='whitesmoke', edgecolor='gray', alpha=0.9))
    
    plt.show()

interact(simulate_added_mass,
         added_mass_percentage=widgets.FloatSlider(min=20.0, max=150.0, step=10.0, value=80.0, description='Sway Added %:'));

interactive(children=(FloatSlider(value=80.0, description='Sway Added %:', max=150.0, min=20.0, step=10.0), Ou…

---

## 3. Coriolis & Centripetal Cross-Coupling ($C(\nu)$)

Because our state variables are tracked within the vessel's moving, turning **BODY frame** instead of a static map frame, we must mathematically handle the Coriolis effect. 

Whenever a vessel has forward speed ($u$) and begins to swing with a yaw turn rate ($r$), a phantom cross-coupling force ($m \cdot u \cdot r$) instantly pushes the hull sideways. This is the exact physics behind why a ship drifts outward when turning hard.

In [3]:
def simulate_coriolis(forward_speed_knots, turn_rate_deg_sec):
    """
    Computes the instantaneous Coriolis cross-coupling force pushing a 
    100-tonne vessel sideways based on its forward speed and turn rate.
    """
    m = 100000.0  # 100 tonnes in kg
    u = forward_speed_knots * 0.51444  # Knots to m/s
    r = np.radians(turn_rate_deg_sec)  # Deg/s to rad/s
    
    # Coriolis Force (Sway direction): Y_coriolis = m * u * r
    f_coriolis_kn = (m * u * r) / 1000.0
    
    print("=" * 60)
    print("             CORIOLIS CROSS-COUPLING ANALYSIS")
    print("=" * 60)
    print(f"  Forward Velocity (u) : {forward_speed_knots} knots ({u:.2f} m/s)")
    print(f"  Turn Rate (r)        : {turn_rate_deg_sec}°/sec ({r:.3f} rad/s)")
    print("-" * 60)
    print(f"  Phantom Sideways Force (Sway) = {f_coriolis_kn:.2f} kN")
    print("-" * 60)
    
    if f_coriolis_kn > 10.0:
        print("⚠️ HIGH DRIFT RISK: Significant lateral force generated.")
        print("  The path-following algorithm must crab into the turn to compensate.")
    else:
        print("✅ Low drift tracking zone.")

interact(simulate_coriolis,
         forward_speed_knots=widgets.FloatSlider(min=0.0, max=30.0, step=2.0, value=15.0, description='Speed (u):'),
         turn_rate_deg_sec=widgets.FloatSlider(min=0.0, max=10.0, step=0.5, value=4.0, description='Turn Rate (r):'));

interactive(children=(FloatSlider(value=15.0, description='Speed (u):', max=30.0, step=2.0), FloatSlider(value…

---

## 4. Hydrodynamic Damping ($D(\nu)$)

Water drag is heavily non-linear. Fossen divides this resistance into two primary elements:
1. **Linear Damping:** Low-speed skin friction where fluid layers slide smoothly past the plating.
2. **Quadratic Damping:** High-speed pressure drag caused by flow separation, vortex shedding, and wave-making energy losses. It scales with the *square* of your speed ($u^2$).

In [4]:
def simulate_damping(linear_coeff, quadratic_coeff):
    """
    Plots the non-linear relationship of hydrodynamic damping, showing how
    quadratic drag dominates skin friction at operational transit speeds.
    """
    u = np.linspace(0, 10, 200) # 0 to ~20 knots
    
    f_linear = linear_coeff * u
    f_quadratic = quadratic_coeff * (u**2)
    f_total = f_linear + f_quadratic
    
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(u, f_linear, color='royalblue', linestyle=':', linewidth=2, label='Linear Skin Friction')
    ax.plot(u, f_quadratic, color='purple', linestyle='--', linewidth=2, label='Quadratic Pressure Drag')
    ax.plot(u, f_total, color='crimson', linewidth=3, label='Total Damping D(u)u')
    
    ax.set_title("Kinetics Lab: Damping Force vs. Vessel Speed", fontsize=12)
    ax.set_xlabel("Vessel Velocity (u) [m/s]", fontsize=10)
    ax.set_ylabel("Resisting Hydrodynamic Force [Newtons]", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    ax.axvspan(0, 2, color='green', alpha=0.1, label='Low-Speed Docking (Linear Dominant)')
    ax.legend(loc='upper left')
    plt.show()

interact(simulate_damping,
         linear_coeff=widgets.FloatSlider(min=10.0, max=200.0, step=10.0, value=50.0, description='Linear (Dl):'),
         quadratic_coeff=widgets.FloatSlider(min=5.0, max=100.0, step=5.0, value=30.0, description='Quad (Dq):'));

interactive(children=(FloatSlider(value=50.0, description='Linear (Dl):', max=200.0, min=10.0, step=10.0), Flo…

---

## 5. Restoring Forces and Moments ($g(\eta)$)

The vector $g(\eta)$ accounts for the hydrostatic forces of gravity and buoyancy. In a 3-DOF horizontal model (Surge, Sway, Yaw), restoring forces are zero because water doesn't naturally push a ship back to a specific GPS coordinate.

However, in vertical/rotational channels (**Roll, Pitch, Heave**), hydrostatics act as a natural spring system:

$$g(\eta) = \begin{bmatrix}
0 \\
0 \\
(\rho g \nabla - m g) \\
\rho g \nabla (z_b - z_g) \sin(\phi) \\
\rho g \nabla (z_b - z_g) \sin(\theta) \\
0
\end{bmatrix}$$

Where:
* $\nabla$ is the volume of displaced water.
* $z_b$ is the vertical position of the Center of Buoyancy.
* $z_g$ is the vertical position of the Center of Gravity.
* $\phi$ and $\theta$ are the Roll and Pitch angles.

For a submerged submarine to be stable, the Center of Gravity must sit **below** the Center of Buoyancy ($z_g > z_b$ in SNAME), creating a positive metacentric restoring torque that fights against roll and pitch errors.

In [2]:
def simulate_restoring_torque(bg_distance_meters, pitch_angle_deg):
    """
    Simulates the hydrostatic restoring torque (metacentric stability) 
    acting on a submerged submarine trying to fight a pitch displacement.
    """
    # Baseline physical constants for a small submarine
    vessel_mass_kg = 50000.0  # 50 Tonnes
    g = 9.81                  # m/s²
    W = vessel_mass_kg * g    # Total Weight Force (Newtons)
    
    # Convert pitch angle to radians
    theta_rad = np.radians(pitch_angle_deg)
    
    # Restoring Torque: M_restoring = -W * BG * sin(theta)
    # BG is the distance between Center of Buoyancy and Center of Gravity
    torque_nm = -W * bg_distance_meters * np.sin(theta_rad)
    
    print("=" * 70)
    print("               HYDROSTATIC RESTORING MOMENT ANALYSIS")
    print("=" * 70)
    print(f"  Vessel Displacement Weight (W) : {W:,.0f} Newtons")
    print(f"  Metacentric Distance (BG)     : {bg_distance_meters:.2f} meters (G below B)")
    print(f"  Current Pitch Error (theta)   : {pitch_angle_deg:+.1f} degrees")
    print("-" * 70)
    print(f"  Generated Restoring Torque    : {torque_nm / 1000.0:.2f} kN·m")
    print("-" * 70)
    
    if bg_distance_meters <= 0.0:
        print("🚨 CRITICAL STABILITY FAILURE:")
        print("  The Center of Gravity is above or equal to the Center of Buoyancy!")
        print("  The vessel has zero righting energy and will capsizingly roll or turtle.")
    elif abs(pitch_angle_deg) > 0.0:
        print("✅ STABLE RIGHTING MOMENT:")
        print(f"  The hydrostatics are actively generating {-torque_nm / 1000.0:.2f} kN·m of torque")
        print("  to push the vessel's hull back toward a flat, neutral trim.")
    else:
        print("✅ Vessel is in perfectly level static equilibrium.")

interact(simulate_restoring_torque,
         bg_distance_meters=widgets.FloatSlider(min=-0.5, max=2.0, step=0.1, value=0.5, description='BG Dist (m):'),
         pitch_angle_deg=widgets.FloatSlider(min=-45.0, max=45.0, step=2.0, value=15.0, description='Pitch Err:'));

interactive(children=(FloatSlider(value=0.5, description='BG Dist (m):', max=2.0, min=-0.5), FloatSlider(value…

## Summary of Chapter 3 Physics

1. **Newton-Euler Formulations** derive the dry vessel dynamics ($M_{RB}$, $C_{RB}$) by tracking linear and angular momentum balances about an arbitrary vehicle origin.
2. **Total Inertia ($M$)** unifies structural mass with hydrodynamic fluid displacement ($M = M_{RB} + M_A$), where added mass acts as a direction-dependent penalty based on hull geometry.
3. **Coriolis Forces ($C(\nu)\nu$)** use skew-symmetric matrices to map the velocity-dependent "phantom" forces born from computing motion inside a rotating, non-inertial BODY frame, inducing heavy lateral drift during turns.
4. **Damping ($D(\nu)\nu$)** models fluid resistance, transitioning from linear skin friction during low-speed maneuvers to non-linear quadratic pressure drag at transit speeds.
5. **Restoring Forces ($g(\eta)$)** capture the hydrostatic equilibrium between gravity and buoyancy, acting as a natural mechanical pendulum that dictates the rotational stability (metacentric height) of the hull.